In [2]:
import tensorflow as tf
import keras

print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)

from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, BatchNormalization, Activation, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

print("TensorFlow Keras imports: OK")

TensorFlow: 2.21.0
Keras: 3.12.1
TensorFlow Keras imports: OK


In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from category_encoders import TargetEncoder

from tensorflow.keras import Sequential
from tensorflow.keras.layers import (
    Dense,
    BatchNormalization,
    Activation,
    Dropout
)
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    ReduceLROnPlateau
)


In [3]:
df =pd.read_csv("porter.csv")
df

,market_id,created_at,actual_delivery_time,store_id,store_primary_category,order_protocol,total_items,subtotal,num_distinct_items,min_item_price,max_item_price,total_onshift_partners,total_busy_partners,total_outstanding_orders
0,1.0,2015-02-06 22:24:17,2015-02-06 23:27:16,df263d996281d984952c07998dc54358,american,1.0,4,3441,4,557,1239,33.0,14.0,21.0
1,2.0,2015-02-10 21:49:25,2015-02-10 22:56:29,f0ade77b43923b38237db569b016ba25,mexican,2.0,1,1900,1,1400,1400,1.0,2.0,2.0
2,3.0,2015-01-22 20:39:28,2015-01-22 21:09:09,f0ade77b43923b38237db569b016ba25,NaN,1.0,1,1900,1,1900,1900,1.0,0.0,0.0
3,3.0,2015-02-03 21:21:45,2015-02-03 22:13:00,f0ade77b43923b38237db569b016ba25,NaN,1.0,6,6900,5,600,1800,1.0,1.0,2.0
4,3.0,2015-02-15 02:40:36,2015-02-15 03:20:26,f0ade77b43923b38237db569b016ba25,NaN,1.0,3,3900,3,1100,1600,6.0,6.0,9.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
197423,1.0,2015-02-17 00:19:41,2015-02-17 01:24:48,a914ecef9c12ffdb9bede64bb703d877,fast,4.0,3,1389,3,345,649,17.0,17.0,23.0
197424,1.0,2015-02-13 00:01:59,2015-02-13 00:58:22,a914ecef9c12ffdb9bede64bb703d877,fast,4.0,6,3010,4,405,825,12.0,11.0,14.0
197425,1.0,2015-01-24 04:46:08,2015-01-24 05:36:16,a914ecef9c12ffdb9bede64bb703d877,fast,4.0,5,1836,3,300,399,39.0,41.0,40.0
197426,1.0,2015-02-01 18:18:15,2015-02-01 19:23:22,c81e155d85dae5430a8cee6f2242e82c,sandwich,1.0,1,1175,1,535,535,7.0,7.0,12.0


In [4]:
# Convert timestamps
df['created_at'] = pd.to_datetime(
    df['created_at'],
    errors='coerce'
)

df['actual_delivery_time'] = pd.to_datetime(
    df['actual_delivery_time'],
    errors='coerce'
)

# Delivery time in minutes
df['delivery_time'] = (
    df['actual_delivery_time'] -
    df['created_at']
).dt.total_seconds() / 60

In [5]:
df = df.dropna(
    subset=['delivery_time']
).copy()

In [6]:
# Categorical columns
categorical_cols = [
    'store_id',
    'store_primary_category',
    'order_protocol'
]

for col in categorical_cols:
    df[col] = df[col].fillna(
        df[col].mode()[0]
    )


# Numerical columns
numerical_cols = [
    'total_items',
    'subtotal',
    'num_distinct_items',
    'min_item_price',
    'max_item_price',
    'total_onshift_partners',
    'total_busy_partners',
    'total_outstanding_orders'
]

for col in numerical_cols:
    df[col] = df[col].fillna(
        df[col].median()
    )

In [7]:
X = df.drop(
    columns=[
        'delivery_time',
        'actual_delivery_time'
    ]
)

y = df['delivery_time']

In [8]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42
)

In [9]:
target_encode_cols = [
    'store_id',
    'store_primary_category'
]

encoder = TargetEncoder(
    cols=target_encode_cols,
    handle_missing='value',
    handle_unknown='value'
)

X_train = encoder.fit_transform(
    X_train,
    y_train
)

X_val = encoder.transform(
    X_val
)

X_test = encoder.transform(
    X_test
)

In [10]:
X_train = pd.get_dummies(
    X_train,
    columns=['order_protocol'],
    dtype=float
)

X_val = pd.get_dummies(
    X_val,
    columns=['order_protocol'],
    dtype=float
)

X_test = pd.get_dummies(
    X_test,
    columns=['order_protocol'],
    dtype=float
)

In [11]:
X_val = X_val.reindex(
    columns=X_train.columns,
    fill_value=0
)

X_test = X_test.reindex(
    columns=X_train.columns,
    fill_value=0
)

In [12]:
print("Training features:", X_train.shape)
print("Validation features:", X_val.shape)
print("Testing features:", X_test.shape)

print("\nFeatures:")
print(X_train.columns.tolist())

Training features: (157936, 19)
Validation features: (19742, 19)
Testing features: (19743, 19)

Features:
['market_id', 'created_at', 'store_id', 'store_primary_category', 'total_items', 'subtotal', 'num_distinct_items', 'min_item_price', 'max_item_price', 'total_onshift_partners', 'total_busy_partners', 'total_outstanding_orders', 'order_protocol_1.0', 'order_protocol_2.0', 'order_protocol_3.0', 'order_protocol_4.0', 'order_protocol_5.0', 'order_protocol_6.0', 'order_protocol_7.0']


In [16]:
X_train

,market_id,created_at,store_id,store_primary_category,total_items,subtotal,num_distinct_items,min_item_price,max_item_price,total_onshift_partners,total_busy_partners,total_outstanding_orders,order_protocol_1.0,order_protocol_2.0,order_protocol_3.0,order_protocol_4.0,order_protocol_5.0,order_protocol_6.0,order_protocol_7.0
70519,3.0,2015-02-07 19:26:17,49.485560,47.932626,11,3524,6,194,799,2.0,2.0,2.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
50630,1.0,2015-02-09 02:58:23,49.997940,47.932626,3,3500,2,1150,1200,59.0,60.0,102.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
57321,6.0,2015-02-15 21:22:57,48.173063,46.702348,2,1200,2,500,700,37.0,34.0,41.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
52856,2.0,2015-02-14 03:38:34,43.743499,49.971723,3,2675,3,775,1075,59.0,57.0,111.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
35924,6.0,2015-02-08 18:07:22,47.877528,52.596462,3,3200,2,300,750,37.0,34.0,41.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119883,2.0,2015-01-25 21:19:54,45.746299,47.932626,2,2774,2,599,2175,71.0,77.0,44.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
103697,4.0,2015-02-13 00:49:15,47.202439,46.025668,1,1375,1,975,975,33.0,24.0,23.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
131936,1.0,2015-02-12 02:39:43,39.793044,50.871233,6,5970,6,895,1195,127.0,85.0,142.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
146872,4.0,2015-01-24 04:19:52,42.817629,46.735644,1,1395,1,1395,1395,45.0,52.0,90.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0


In [14]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

DTypePromotionError: The DType <class 'numpy.dtypes.DateTime64DType'> could not be promoted by <class 'numpy.dtypes.Float64DType'>. This means that no common DType exists for the given inputs. For example they cannot be stored in a single array unless the dtype is `object`. The full list of DTypes is: (<class 'numpy.dtypes.Float64DType'>, <class 'numpy.dtypes.DateTime64DType'>, <class 'numpy.dtypes.Float64DType'>, <class 'numpy.dtypes.Float64DType'>, <class 'numpy.dtypes.Int64DType'>, <class 'numpy.dtypes.Int64DType'>, <class 'numpy.dtypes.Int64DType'>, <class 'numpy.dtypes.Int64DType'>, <class 'numpy.dtypes.Int64DType'>, <class 'numpy.dtypes.Float64DType'>, <class 'numpy.dtypes.Float64DType'>, <class 'numpy.dtypes.Float64DType'>, <class 'numpy.dtypes.Float64DType'>, <class 'numpy.dtypes.Float64DType'>, <class 'numpy.dtypes.Float64DType'>, <class 'numpy.dtypes.Float64DType'>, <class 'numpy.dtypes.Float64DType'>, <class 'numpy.dtypes.Float64DType'>, <class 'numpy.dtypes.Float64DType'>)

In [ ]:
y_train = y_train.to_numpy()
y_val = y_val.to_numpy()
y_test = y_test.to_numpy()

In [ ]:
L2Reg = tf.keras.regularizers.L2(
    l2=1e-4
)

model = Sequential([
    
    Dense(
        256,
        kernel_regularizer=L2Reg,
        input_shape=(X_train_scaled.shape[1],)
    ),
    BatchNormalization(),
    Activation("relu"),
    Dropout(0.20),

    Dense(
        128,
        kernel_regularizer=L2Reg
    ),
    BatchNormalization(),
    Activation("relu"),
    Dropout(0.20),

    Dense(
        64,
        kernel_regularizer=L2Reg
    ),
    BatchNormalization(),
    Activation("relu"),
    Dropout(0.20),

    Dense(
        32,
        kernel_regularizer=L2Reg
    ),
    BatchNormalization(),
    Activation("relu"),
    Dropout(0.20),

    Dense(
        1,
        activation="linear"
    )
])

In [ ]:
L2Reg = tf.keras.regularizers.L2(
    l2=1e-4
)

model = Sequential([
    
    Dense(
        256,
        kernel_regularizer=L2Reg,
        input_shape=(X_train_scaled.shape[1],)
    ),
    BatchNormalization(),
    Activation("relu"),
    Dropout(0.20),

    Dense(
        128,
        kernel_regularizer=L2Reg
    ),
    BatchNormalization(),
    Activation("relu"),
    Dropout(0.20),

    Dense(
        64,
        kernel_regularizer=L2Reg
    ),
    BatchNormalization(),
    Activation("relu"),
    Dropout(0.20),

    Dense(
        32,
        kernel_regularizer=L2Reg
    ),
    BatchNormalization(),
    Activation("relu"),
    Dropout(0.20),

    Dense(
        1,
        activation="linear"
    )
])

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="mae",
    metrics=["mae"]
)

In [ ]:
monitor='loss'
mode='max'

In [ ]:
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=10,
    mode='min',
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=4,
    mode='min',
    min_lr=1e-6,
    verbose=1
)

checkpoint = ModelCheckpoint(
    "porter_best.keras",
    monitor='val_loss',
    mode='min',
    save_best_only=True,
    verbose=1
)

In [ ]:
history = model.fit(
    X_train_scaled,
    y_train,

    validation_data=(
        X_val_scaled,
        y_val
    ),

    epochs=100,
    batch_size=36,

    callbacks=[
        early_stopping,
        reduce_lr,
        checkpoint
    ],

    verbose=1
)

In [ ]:
train_loss, train_mae = model.evaluate(
    X_train_scaled,
    y_train,
    verbose=0
)

val_loss, val_mae = model.evaluate(
    X_val_scaled,
    y_val,
    verbose=0
)

test_loss, test_mae = model.evaluate(
    X_test_scaled,
    y_test,
    verbose=0
)

print("Training MAE:", train_mae)
print("Validation MAE:", val_mae)
print("Testing MAE:", test_mae)

In [ ]:
import joblib

# Save model
model.save(
    "porter_delivery_model.keras"
)

# Save scaler
joblib.dump(
    scaler,
    "porter_scaler.pkl"
)

# Save target encoder
joblib.dump(
    encoder,
    "porter_target_encoder.pkl"
)

# Save feature columns
joblib.dump(
    X_train.columns.tolist(),
    "porter_feature_columns.pkl"
)

print("All deployment files saved successfully!")